#### IMPORTAR LIBRERIAS Y CARGAR LOS DATOS NECESARIOS

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import KeyedVectors
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Concatenate
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import re
import gensim.downloader as api
from keras.layers import LSTM
import torch
from transformers import pipeline
from datasets import load_dataset
from transformers import AutoTokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer, TrainingArguments, GPT2Tokenizer
from tensorflow.keras.models import load_model


In [7]:
def limpiar_agencia(texto):
    return re.sub(r'^[A-Z\s/\-]+ \([A-Za-z]+\)\s*[-–—]\s*', '', texto)

In [8]:
df_true = pd.read_csv('data/True.csv')
df_fake = pd.read_csv('data/Fake.csv')

df_true['output'] = 'True'
df_fake['output'] = 'Fake'

data = pd.concat([df_true, df_fake], ignore_index=True)
data = data.drop_duplicates()

data['text'] = data['text'].astype(str).apply(limpiar_agencia)

data.head()

,title,text,subject,date,output
0,"As U.S. budget fight looms, Republicans flip t...",The head of a conservative Republican faction ...,politicsNews,"December 31, 2017",True
1,U.S. military to accept transgender recruits o...,Transgender people will be allowed for the fir...,politicsNews,"December 29, 2017",True
2,Senior U.S. Republican senator: 'Let Mr. Muell...,The special counsel investigation of links bet...,politicsNews,"December 31, 2017",True
3,FBI Russia probe helped by Australian diplomat...,Trump campaign adviser George Papadopoulos tol...,politicsNews,"December 30, 2017",True
4,Trump wants Postal Service to charge 'much mor...,President Donald Trump called on the U.S. Post...,politicsNews,"December 29, 2017",True


In [18]:
# Extraer variables
X1 = data['text'].astype(str)
y = np.array(data['output'])

In [19]:
# Tokenizer
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X1 )

# Secuencias numéricas
sequences1 = tokenizer.texts_to_sequences(X1)

# Padding
max_len = 5000
X1_padded = pad_sequences(sequences1, maxlen=max_len)

#### RESUMIR NOTICIAS

Para obtener una versión condensada de las noticias, hemos utilizado un modelo preentrenado de la libreria transformers de HuggingFace. En concreto, hemos utilizado el modelo "facebook/bart-large-cnn", que está especializado en tareas de resumen automático de textos largos.

In [4]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use mps:0


Por ejemplo, en este caso aplicamos el pipeline de resumen automático (summarization) al primer texto del dataset de noticias, y generamos un resumen con una longitud controlada de entre 30 y 130 palabras. 

In [9]:
resumen = summarizer(data['text'].iloc[0], max_length=130, min_length=30, do_sample=False)[0]['summary_text']
print(resumen)

U.S. Representative Mark Meadows calls himself a "fiscal conservative" on "Face the Nation" Meadows was among Republicans who voted in late December for a debt-financed tax overhaul. The tax bill is expected to balloon the federal budget deficit and add about $1.5 trillion over 10 years.


#### GENERACIÓN DE NOTICIAS FAKE

Preparamos el dataset previamente cargado de fake news (df_fake) eliminando saltos de línea y espacios innecesarios. A cada texto le añadimos los tokens especiales <|startoftext|> y <|endoftext|> requeridos por el modelo GPT-2 para entender los límites de cada muestra y lo guardamos en un csv.

In [10]:
# Limpieza y formato
def limpiar_texto(txt):
    return txt.replace("\n", " ").replace("\r", " ").strip()

fake_lines = df_fake['text'].astype(str).apply(limpiar_texto)

# Añadir tokens especiales requeridos por GPT-2
fake_lines = fake_lines.apply(lambda x: "<|startoftext|>" + x + "<|endoftext|>")

# Guardar a archivos
fake_lines.to_csv("fake_news.txt", index=False, header=False)

##### Entrenamiento del modelo

Entrenamos ahora una versión del modelo GPT-2 utilizando el dataset de noticias falsas preprocesado. Con ello buscamos que el modelo aprenda a generar nuevos textos que imiten el estilo y contenido de las noticias falsas originales.
Tokenizamos los textos utilizando el tokenizador de GPT-2. Después utilizamos DataCollatorForLanguageModeling para preparar los datos antes de entrenar el modelo añadiendo padding automáticamente. Finalmente entrenamos y guardamos el modelo.

In [ ]:
# 1. Cargar dataset
dataset = load_dataset("text", data_files={"train": "fake_news.txt"})

# 2. Cargar tokenizer y modelo
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 no tiene padding por defecto

model = GPT2LMHeadModel.from_pretrained("gpt2")

# 3. Tokenización
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-fake",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=100,
    fp16=False,
    push_to_hub=False,
)

# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 7. Entrenar
trainer.train()

# 8. Guardar modelo
trainer.save_model("./gpt2-fake")
tokenizer.save_pretrained("./gpt2-fake")

Se carga el modelo GPT-2 previamente ajustado (fine-tuned) con datos de noticias falsas, junto con su tokenizer. Luego, se crea un pipeline de generación de texto que permite obtener nuevas noticias a partir de un texto inicial.

In [11]:
# Cargar el modelo y tokenizer desde la carpeta
model = GPT2LMHeadModel.from_pretrained("./gpt2-fake")
gpt_tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-fake")

# Crear el generador de texto
generator = pipeline("text-generation", model=model, tokenizer=gpt_tokenizer)

Device set to use mps:0


Una vez cargado el modelo y creado el pipeline de generación de texto, se utiliza la función generator para producir una noticia falsa. En eset ejemplo partimos de una frase inicial o prompt "The government announced today that", y el modelo completa el texto de forma coherente basándose en lo aprendido durante el entrenamiento.

In [12]:
fake_news = generator(
    "The government announced today that",            # prompt
    max_length=100,         # longitud total
    num_return_sequences=1, # cantidad de muestras
    do_sample=True,         # sampling activado para creatividad
    temperature=0.9,        # controla aleatoriedad
)[0]['generated_text']

print(fake_news)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


The government announced today that former Secretary of State Hillary Clinton will be hosting a free gala at the New York Hilton in honor of National Rifle Association founder, Wayne LaPierre.Former Secretary of State, Hillary Clinton is expected to host the first-ever gala honoring NRA founder and former President,  Wayne LaPierre said of the event. The NRA s vice president of government affairs, Wayne LaPierre, said the event is part of a larger effort to honor and to promote  public service. This


Después de generar la noticia falsa, utilizamos el modelo LSTM previamente entrenado para determinar si el texto tiene característica de noticia falsa o verdadera. 
Se generan una serie de noticias falsas utilizando el modelo GPT-2 fine-tuned, y cada una se evalúa con el modelo LSTM para determinar si puede ser clasificada como una noticia falsa.

In [13]:
# Cargar el modelo entrenado
model_clasificador = load_model('modelo_LSTM_Word2Vec.keras')

In [16]:
def predict_news(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prob = model_clasificador.predict(padded)[0][0]
    label = "Real" if prob > 0.5 else "Fake"
    print(f"Predicted: {label} (Probability: {prob:.4f})")

In [21]:
for i in range(5):
    generated = generator("Donald Trump", max_length=100, do_sample=True, temperature=0.9)[0]['generated_text']
    print(f"\n Noticia {i+1}:\n{generated}")
    predict_news(generated)


 Noticia 1:
Donald Trump tweeted out the words in support of the president of the United States during a campaign rally.The phrase is not just offensive, it also serves to indicate exactly why we are in this race to make a complete basket of deplorables on both the right and the left. Here is Trump s tweet:The words that get said in the campaign are totally acceptable when it s used as a noun. However, Trump s supporters see their words as being totally illegal. If these words were actually used
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step
Predicted: Fake (Probability: 0.0295)

 Noticia 2:
Donald Trump supporters are not your average Trump fan when it comes to the Republican presidential nominee. He hates things, and the very type that he hates. So when he told a rally in North Carolina (the home of the Republican National Convention) that only 2 months ago, that there would be a  Great Storm,  it just might be a sign that Trump is no longer even able to handle the facts on things.He said, 